# 05 — System Integration Benchmark

End-to-end benchmark of the full LLM-driven agent system.

**Tests**:
1. **Step latency waterfall** — DB query → memory read → LLM decision → memory write → tracker log
2. **Scalability** — N = 10, 50, 100, 500 agents
3. **Humanistic scoring** — trajectory patterns after 200 steps

**5 Humanistic Dimensions** (0-1 each):
- Diversity — unique edges / total steps
- Archetype consistency — archetype-appropriate destination choices
- Amenity plausibility — need state matches visited amenity
- Spatial realism — trajectory entropy
- Decision coherence — reasoning consistency across steps

In [1]:
import asyncio
import math
import sys
import time
import json
import statistics
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

sys.path.insert(0, str(Path('scripts').resolve()))
from bench_helpers import start_server, stop_server, wait_for_server, run_steps, collect_agent_data
import httpx

PROJECT_ROOT = Path(r'D:\IaaC\2ND_YEAR\THESIS\LLM_Based_UrbanABM')
BASE_URL = 'http://127.0.0.1:8000'

sns.set_theme(style='whitegrid', palette='muted')
np.random.seed(42)

In [2]:
# --- Step Latency: LLM-driven vs Rule-based ---
N_STEPS = 50

async def bench_step_latency():
    all_timings = []

    for mode, env_overrides in [
        ('LLM-driven', {'LLM_CALLS_PER_STEP': '-1', 'NUM_AGENTS': '50'}),
        ('Rule-based', {'LLM_CALLS_PER_STEP': '0', 'NUM_AGENTS': '50'}),
    ]:
        print(f'\n=== Benchmarking {mode} mode ===')
        proc = start_server(env_overrides=env_overrides)
        try:
            ready = await wait_for_server(timeout=90)
            if not ready:
                print(f'  Server failed to start for {mode} mode')
                continue

            print(f'  Server ready. Running {N_STEPS} steps...')
            step_results = await run_steps(N_STEPS)

            for r in step_results:
                all_timings.append({
                    'step': r['step'],
                    'latency_ms': r['latency_ms'],
                    'mode': mode,
                })

            total_errors = step_results[-1]['llm_stats'].get('total_errors', 0)
            if mode == 'LLM-driven' and total_errors > 0:
                print(f'  WARNING: {total_errors} LLM errors — results may reflect fallback behavior')
            print(f'  Collected {len(step_results)} step timings')
        finally:
            stop_server(proc)

    return pd.DataFrame(all_timings)

timing_df = await bench_step_latency()
print('\nStep latency summary:')
print(timing_df.groupby('mode')['latency_ms'].describe())


=== Benchmarking LLM-driven mode ===
  Server ready. Running 50 steps...


ReadTimeout: 

In [ ]:
# --- Scalability: varying agent counts ---
AGENT_COUNTS = [10, 50, 100, 500]
STEPS_PER_CONFIG = 10

async def bench_scalability():
    rows = []
    for n_agents in AGENT_COUNTS:
        for mode, llm_budget in [('LLM-driven', '-1'), ('Rule-based', '0')]:
            print(f'\n=== N={n_agents}, {mode} ===')
            env = {'NUM_AGENTS': str(n_agents), 'LLM_CALLS_PER_STEP': llm_budget}
            proc = start_server(env_overrides=env)
            try:
                ready = await wait_for_server(timeout=120)
                if not ready:
                    print(f'  SKIP: server failed to start')
                    continue

                step_results = await run_steps(STEPS_PER_CONFIG)
                avg_ms = statistics.mean(r['latency_ms'] for r in step_results)
                rows.append({
                    'n_agents': n_agents,
                    'step_ms': round(avg_ms, 1),
                    'mode': mode,
                })
                print(f'  avg step: {avg_ms:.1f} ms')
            finally:
                stop_server(proc)

    return pd.DataFrame(rows)

scale_data = await bench_scalability()
print('\nScalability results:')
print(scale_data)

In [ ]:
# --- Humanistic Scoring from real trajectories ---
HUMANISTIC_STEPS = 100
HUMANISTIC_AGENTS = 50

ARCHETYPE_TARGET_TYPES = {
    'tourist':  {'attraction', 'museum', 'cafe', 'restaurant', 'park', 'viewpoint'},
    'resident': {'supermarket', 'bakery', 'park', 'pharmacy'},
    'commuter': {'office', 'coworking_space', 'commercial', 'company', 'bank', 'cafe'},
    'student':  {'university', 'library', 'cafe', 'fast_food', 'park'},
}

COHERENCE_KEYWORDS = {
    'tourist':  ['explore', 'visit', 'see', 'attraction', 'new', 'scenic', 'curious', 'discover'],
    'resident': ['home', 'familiar', 'routine', 'errand', 'grocery', 'pharmacy', 'usual'],
    'commuter': ['direct', 'efficient', 'work', 'shortest', 'fast', 'destination', 'transit'],
    'student':  ['campus', 'library', 'study', 'friend', 'cafe', 'budget', 'social'],
}


def compute_humanistic_scores(archetype, profile, memory, stream_events):
    visited_edges = memory.get('visited_edges', {})
    total_visits = sum(visited_edges.values()) if visited_edges else 1
    unique_edges = len(visited_edges)
    diversity = min(1.0, unique_edges / max(1, total_visits))

    visited_amenities = memory.get('visited_amenities', [])
    expected = ARCHETYPE_TARGET_TYPES.get(archetype, set())
    if visited_amenities:
        matching = sum(1 for a in visited_amenities if a.get('type', '') in expected)
        archetype_consistency = matching / len(visited_amenities)
    else:
        archetype_consistency = 0.5

    needs_events = [e for e in stream_events if e.get('topic') == 'needs']
    amenity_events = [e for e in stream_events if e.get('topic') == 'amenity_visit']
    if amenity_events and needs_events:
        plausible_count = 0
        for ae in amenity_events:
            ae_step = ae.get('step', 0)
            prior_needs = [n for n in needs_events if n.get('step', 0) <= ae_step]
            if not prior_needs:
                continue
            needs_meta = prior_needs[-1].get('metadata', {})
            high_needs = [k for k, v in needs_meta.items()
                         if isinstance(v, (int, float)) and v > 0.5]
            if high_needs:
                plausible_count += 1
        amenity_plausibility = plausible_count / len(amenity_events)
    else:
        amenity_plausibility = 0.5

    if visited_edges and len(visited_edges) > 1:
        counts = list(visited_edges.values())
        total = sum(counts)
        probs = [c / total for c in counts]
        entropy = -sum(p * math.log2(p) for p in probs if p > 0)
        max_entropy = math.log2(len(counts))
        spatial_realism = entropy / max_entropy if max_entropy > 0 else 0
    else:
        spatial_realism = 0.0

    mob_events = [e for e in stream_events if e.get('topic') == 'mobility']
    keywords = COHERENCE_KEYWORDS.get(archetype, [])
    if mob_events and keywords:
        coherent = sum(1 for me in mob_events
                       if any(kw in (me.get('description', '') or '').lower()
                              for kw in keywords))
        decision_coherence = coherent / len(mob_events)
    else:
        decision_coherence = 0.5

    return {
        'diversity': round(diversity, 3),
        'archetype_consistency': round(archetype_consistency, 3),
        'amenity_plausibility': round(amenity_plausibility, 3),
        'spatial_realism': round(spatial_realism, 3),
        'decision_coherence': round(decision_coherence, 3),
    }


async def run_and_score():
    all_scores = []

    for mode, llm_budget in [('LLM-driven', '-1'), ('Rule-based', '0')]:
        print(f'\n=== Humanistic scoring: {mode} ===')
        env = {
            'NUM_AGENTS': str(HUMANISTIC_AGENTS),
            'LLM_CALLS_PER_STEP': llm_budget,
            'SPAWN_SEED': '42',
        }
        proc = start_server(env_overrides=env)
        try:
            ready = await wait_for_server(timeout=90)
            if not ready:
                print(f'  SKIP: server failed to start')
                continue

            print(f'  Running {HUMANISTIC_STEPS} steps...')
            step_results = await run_steps(HUMANISTIC_STEPS)

            total_errors = step_results[-1]['llm_stats'].get('total_errors', 0)
            if mode == 'LLM-driven' and total_errors > 0:
                print(f'  WARNING: {total_errors} LLM errors detected')

            async with httpx.AsyncClient(base_url=BASE_URL, timeout=120) as client:
                resp = await client.get('/api/agents')
                features = resp.json()['features']
                agent_ids = [f['properties']['id'] for f in features]

            print(f'  Collecting data from {len(agent_ids)} agents...')
            agent_data = await collect_agent_data(agent_ids)

            for aid, data in agent_data.items():
                mem = data['memory']
                stream = data['stream']
                profile = mem.get('agent_profile', {})
                archetype = profile.get('archetype', 'unknown')

                s = compute_humanistic_scores(archetype, profile, mem, stream)
                s['archetype'] = archetype
                s['mode'] = mode
                all_scores.append(s)
        finally:
            stop_server(proc)

    return pd.DataFrame(all_scores)

scores = await run_and_score()
dims = ['diversity', 'archetype_consistency', 'amenity_plausibility',
        'spatial_realism', 'decision_coherence']
print(scores.groupby('mode')[dims].mean())

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# Step latency over time
for mode, grp in timing_df.groupby('mode'):
    axes[0,0].plot(grp['step'], grp['latency_ms'], label=mode, alpha=0.8)
axes[0,0].set_title('Step Latency Over Time (50 steps)')
axes[0,0].set_xlabel('Step')
axes[0,0].set_ylabel('Latency (ms)')
axes[0,0].legend()

# Scalability
for mode, grp in scale_data.groupby('mode'):
    axes[0,1].plot(grp['n_agents'], grp['step_ms'], marker='o', label=mode)
axes[0,1].set_title('Step Latency vs Agent Count')
axes[0,1].set_xlabel('Number of Agents')
axes[0,1].set_ylabel('Step Time (ms)')
axes[0,1].legend()

# Radar chart — humanistic dimensions
mean_scores = scores.groupby('mode')[dims].mean()
n = len(dims)
angles = [i / n * 2 * np.pi for i in range(n)] + [0]
ax_radar = plt.subplot(2, 2, 3, polar=True)
for mode_name, row in mean_scores.iterrows():
    vals = row.tolist() + [row.iloc[0]]
    ax_radar.plot(angles, vals, label=mode_name)
    ax_radar.fill(angles, vals, alpha=0.1)
ax_radar.set_xticks(angles[:-1])
ax_radar.set_xticklabels([d.replace('_', '\n') for d in dims], size=8)
ax_radar.set_title('Humanistic Dimensions Radar')
ax_radar.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1))

# Bar per archetype — archetype consistency
arch_scores = scores.groupby(['archetype', 'mode'])['archetype_consistency'].mean().unstack()
arch_scores.plot(kind='bar', ax=axes[1,1])
axes[1,1].set_title('Archetype Consistency by Agent Type')
axes[1,1].set_ylabel('Score')
axes[1,1].set_ylim(0, 1)
axes[1,1].tick_params(axis='x', rotation=15)

plt.tight_layout()
plt.savefig('results_05_system_benchmark.png', dpi=150)
plt.show()

In [ ]:
# ── Export results for HTML report ──────────────────────────────────────
import os

os.makedirs('results', exist_ok=True)

export = {
    'timing': timing_df.to_dict(orient='records'),
    'scale': scale_data.to_dict(orient='records'),
    'scores': scores.to_dict(orient='records'),
    'dims': dims,
    'score_summary': scores.groupby('mode')[dims].mean().to_dict(orient='index'),
}

with open('results/05_system.json', 'w') as f:
    json.dump(export, f, indent=2)
print('Exported -> results/05_system.json')